In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_G.xpt"
csv_file = DATA_DIR / "HUQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 HUQ_G XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HUQ010",
    "HUQ020",
    "HUQ030",
    "HUQ040",
    "HUQ050",
    "HUQ060",
    "HUQ071",
    "HUD080",
    "HUQ090"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HUQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9756:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,756 rows x 10 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached HUQ_G.xpt:
#
#     HUQ050 = 1,297 tiny values
#
# Do NOT perform a blanket replacement before
# validating the exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached HUQ_G pattern:
#
#     HUQ050 = 1,297
#
# CDC defines:
#
#     HUQ050 = number of healthcare visits
#
#     0 = None
#
# CDC published frequency:
#
#     0 = 1,297
#

EXPECTED_TINY_COUNTS = {
    "HUQ050": 1297
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated HUQ_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 1297:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed before "
        "validated correction."
    )


print(
    "\nPASS: Exact HUQ050 1,297-value "
    "tiny-zero pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED HUQ050 ZEROS
# ============================================================

#
# Do NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Restore only validated HUQ050 positions.
#

huq050_mask = (
    df["HUQ050"]
    .eq(TINY_VALUE)
)


correction_count = int(
    huq050_mask.sum()
)


if correction_count != 1297:

    raise ValueError(
        "Unexpected HUQ050 tiny-value count."
    )


df.loc[
    huq050_mask,
    "HUQ050"
] = 0


print("\n--- ZERO RESTORATION ---")


print(
    "HUQ050 restored to zero:",
    correction_count
)


if int(
    df["HUQ050"]
    .eq(0)
    .sum()
) != 1297:

    raise ValueError(
        "HUQ050 zero restoration failed."
    )


print(
    "PASS: 1,297 validated HUQ050 zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# numeric source should contain.
#
# ONLY:
#
#     HUQ050 tiny -> 0
#
# is permitted to change.
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["HUQ050"]
    .eq(TINY_VALUE),
    "HUQ050"
] = 0


# ============================================================
# 8. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining after correction:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# IMPORTANT:
#
# Always perform this BEFORE Int64 conversion.
#
# HUQ_G contains NO genuine fractional variables.
#
# If a future/local copy unexpectedly contains
# fractional observations, STOP instead of rounding.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in HUQ_G. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# HUQ_G contains NO genuine decimal fields.
#
# Therefore all 10 columns can safely become
# nullable Int64.
#
# IMPORTANT:
#
# NO .round() is used for conversion.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 10 HUQ_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 12. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 13. CDC CHECK - HUQ010
# ============================================================

#
# General health condition
#
# 1 Excellent       2530
# 2 Very good       2551
# 3 Good            3121
# 4 Fair            1277
# 5 Poor             275
# 7 Refused            1
# 9 Don't know         1
#

check_frequency(
    "HUQ010",
    {
        1: 2530,
        2: 2551,
        3: 3121,
        4: 1277,
        5: 275,
        7: 1,
        9: 1
    },
    0
)


# ============================================================
# 14. CDC CHECK - HUQ020
# ============================================================

#
# Health now compared with one year ago
#

check_frequency(
    "HUQ020",
    {
        1: 1793,
        2: 794,
        3: 6773,
        7: 1,
        9: 2
    },
    393
)


# ============================================================
# 15. CDC CHECK - HUQ030
# ============================================================

#
# Routine place to go for healthcare
#

check_frequency(
    "HUQ030",
    {
        1: 8634,
        2: 1074,
        3: 48
    },
    0
)


# ============================================================
# 16. CDC CHECK - HUQ040
# ============================================================

#
# Type of place most often used for healthcare
#

check_frequency(
    "HUQ040",
    {
        1: 2147,
        2: 6035,
        3: 260,
        4: 152,
        5: 87,
        7: 1
    },
    1074
)


# ============================================================
# 17. CDC CHECK - HUQ050
# ============================================================

#
# Number of healthcare visits during past year.
#
# CDC:
#
# 0 = None             1297
# 1 = 1                2017
# 2 = 2 to 3           2969
# 3 = 4 to 9           2397
# 4 = 10 to 12          529
# 5 = 13 or more        537
# 77 Refused              1
# 99 Don't know           9
# Missing                 0
#
# The attached XPT encoded the 1,297 zeros as
# TINY_VALUE before correction.
#

check_frequency(
    "HUQ050",
    {
        0: 1297,
        1: 2017,
        2: 2969,
        3: 2397,
        4: 529,
        5: 537,
        77: 1,
        99: 9
    },
    0
)


if int(
    df["HUQ050"]
    .eq(0)
    .sum()
) != 1297:

    raise ValueError(
        "HUQ050 validated-zero count changed."
    )


print(
    "PASS: HUQ050 has exactly 1,297 restored zeros."
)


# ============================================================
# 18. CDC CHECK - HUQ060
# ============================================================

#
# Time since last healthcare visit
#
# Only asked for HUQ050=0 participants.
#

check_frequency(
    "HUQ060",
    {
        1: 76,
        2: 86,
        3: 737,
        4: 353,
        5: 36,
        9: 9
    },
    8459
)


# ============================================================
# 19. HUQ050 -> HUQ060 ROUTING CHECK
# ============================================================

#
# HUQ060 is populated for exactly the 1,297 people
# whose HUQ050 response was 0=None.
#

huq050_none = (
    df["HUQ050"]
    .eq(0)
)


huq060_nonmissing = (
    df["HUQ060"]
    .notna()
)


if not np.array_equal(
    huq050_none.to_numpy(),
    huq060_nonmissing.to_numpy()
):

    raise ValueError(
        "HUQ050=0 and HUQ060 routing patterns differ."
    )


print(
    "PASS: HUQ060 routing exactly matches HUQ050=0."
)


# ============================================================
# 20. CDC CHECK - HUQ071
# ============================================================

#
# Overnight hospital patient in last year
#

check_frequency(
    "HUQ071",
    {
        1: 949,
        2: 8805,
        7: 1,
        9: 1
    },
    0
)


# ============================================================
# 21. CDC CHECK - HUD080
# ============================================================

#
# Number of different overnight hospital stays.
#
# CDC:
#
# 1 to 5 = 932 total
# 6       = 17
# Missing = 8807
#
# Exact attached XPT frequencies:
#
# 1 = 704
# 2 = 152
# 3 = 45
# 4 = 23
# 5 = 8
# 6 = 17
#

check_frequency(
    "HUD080",
    {
        1: 704,
        2: 152,
        3: 45,
        4: 23,
        5: 8,
        6: 17
    },
    8807
)


# ============================================================
# 22. HUQ071 -> HUD080 ROUTING CHECK
# ============================================================

#
# HUD080 is populated exactly when HUQ071=1.
#

hospital_yes = (
    df["HUQ071"]
    .eq(1)
)


hud080_nonmissing = (
    df["HUD080"]
    .notna()
)


if not np.array_equal(
    hospital_yes.to_numpy(),
    hud080_nonmissing.to_numpy()
):

    raise ValueError(
        "HUQ071=1 and HUD080 nonmissing "
        "positions differ."
    )


print(
    "PASS: HUD080 routing exactly matches HUQ071=1."
)


# ============================================================
# 23. CDC CHECK - HUQ090
# ============================================================

#
# Seen/talked to mental health professional
# during past 12 months.
#
# Asked age >= 4.
#

check_frequency(
    "HUQ090",
    {
        1: 663,
        2: 7921,
        7: 2,
        9: 1
    },
    1169
)


# ============================================================
# 24. ALLOWED RELEASED CODE CHECK
# ============================================================

#
# Additional corruption safeguard.
#

ALLOWED_VALUES = {

    "HUQ010": {
        1, 2, 3, 4, 5, 7, 9
    },

    "HUQ020": {
        1, 2, 3, 7, 9
    },

    "HUQ030": {
        1, 2, 3
    },

    "HUQ040": {
        1, 2, 3, 4, 5, 7
    },

    "HUQ050": {
        0, 1, 2, 3, 4, 5, 77, 99
    },

    "HUQ060": {
        1, 2, 3, 4, 5, 9
    },

    "HUQ071": {
        1, 2, 7, 9
    },

    "HUD080": {
        1, 2, 3, 4, 5, 6
    },

    "HUQ090": {
        1, 2, 7, 9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected released codes: "
            f"{sorted(unexpected)}"
        )


    print(
        f"PASS: {col} contains only expected codes."
    )


# ============================================================
# 25. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY expected numeric differences from the raw
# pandas-decoded XPT are:
#
#     HUQ050
#     1,297 TINY_VALUE -> 0
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 10 variables match the expected "
    "corrected XPT exactly."
)


# ============================================================
# 26. EXPORT CSV
# ============================================================

#
# No:
#
#     rounding
#
# No:
#
#     float_format=
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 27. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 28. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 10 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 29. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 30. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 31. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 32. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# HUQ_G contains NO genuine decimal variables.
#
# Therefore no populated field should end with ".0".
#
# This global check is safe for HUQ_G specifically.
#
# DO NOT apply it globally to Dietary files or to
# Lab/Exam/Questionnaire files with real decimal values.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 33. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 34. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 35. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "HUQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    1
)


print(
    "HUQ050 validated zero restorations:",
    correction_count
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 HUQ_G XPT -> CSV
Rows: 9,756
Columns: 10
PASS: Structure = 9,756 rows x 10 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 1,297

Tiny values by variable:
HUQ050    1297

PASS: Exact HUQ050 1,297-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
HUQ050 restored to zero: 1297
PASS: 1,297 validated HUQ050 zeros restored.

Tiny values remaining after correction: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
HUQ010: 0 fractional observations
HUQ020: 0 fractional observations
HUQ030: 0 fractional observations
HUQ040: 0 fractional observations
HUQ050: 0 fractional observations
HUQ060: 0 fractional observations
HUQ071: 0 fractional observations
HUD080: 0 fractional observations
HUQ090: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.

PASS: All